# 18 — Paired inference and supplementary Table S20 exports

CPU stage. Freeze thread counts before numerical imports; a small CPU allocation suffices.
Primary: image minus no-image in single-pass RAG, per backbone and source/gallery stratum.
Secondary: the same input contrast within the three revision/gated strategies.
Patient/source-group bootstrap CIs and paired permutations use 10,000 draws, with Holm
correction across selected model/stratum contrasts per outcome and separate primary/secondary families.
Positive differences mean worse FER/omission, better F1. Do not pool duplicate studies across galleries.

Outputs are new supplementary tables and a manuscript-insertion draft, NOT edits to the current
Word files. Smoke runs get a separate technical summary with no publication prose.

In [ ]:
import os
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[name] = "1"

In [ ]:
from pathlib import Path
import os, sys, json

candidate = Path(os.environ.get("JAMIA_RERUN_DIR", Path.cwd())).resolve()
if not (candidate / "rerun_config.json").is_file() and (candidate.parent / "rerun_config.json").is_file():
    candidate = candidate.parent
RERUN_DIR = candidate
if not (RERUN_DIR / "rerun_config.json").is_file():
    raise FileNotFoundError("Copy rerun_config.example.json to rerun_config.json, configure paths, and set JAMIA_RERUN_DIR to the repository root")
implementation = (RERUN_DIR / "src" / "rerun_code").resolve()
sys.path[:] = [entry for entry in sys.path if Path(entry or ".").resolve() != implementation]
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config
_, CONFIG = load_config(RERUN_DIR)
RESULTS_ROOT = Path(os.environ.get("JAMIA_OUTPUT_ROOT", CONFIG["output_root"]))
CONFIG["output_root"] = str(RESULTS_ROOT)
RUN_NAME = os.environ.get("JAMIA_ABLATION_RUN", "matched_input_v1")
ABLATION_ROOT = RESULTS_ROOT / "post_rerun" / "modality_ablation" / RUN_NAME
print("Code:", RERUN_DIR)
print("Existing inputs:", RESULTS_ROOT)
print("NEW output:", ABLATION_ROOT)


In [ ]:
from rerun_code.modality_ablation_statistics import analyze
destination, contrasts = analyze(RERUN_DIR, CONFIG, RUN_NAME)
print("Analysis saved to:", destination)
display(contrasts[["model_key", "bundle", "source_dataset", "condition", "metric",
                   "arm_a", "arm_b", "difference_b_minus_a", "difference_ci_low",
                   "difference_ci_high", "p_holm", "positive_difference_means"]])
print("Download the entire run folder for the manuscript update:")
print(ABLATION_ROOT)